In [241]:
import pandas as pd

data_path = '../results.csv'
df = pd.read_csv(data_path, low_memory=False)

In [242]:
# Transformación y limpieza: NewRole ✅

import numpy as np

mapeo = {
    'I have neither consider or transitioned into a new career or industry': 0,
    'I have somewhat considered changing my career and/or the industry I work in': 1,
    'I have strongly considered changing my career and/or the industry I work in': 1,
    'I have transitioned into a new career and/or industry voluntarily': 1,             
    'I have transitioned into a new career and/or industry involuntarily': np.nan      # La salida fue forzada, no motivada por frustración con la IA.
}

# Hago el mapeo
df['NewRole_binary'] = df['NewRole'].map(mapeo)

# Como NewRole es la variable objetivo, no puedo imputar datos ya que sesgaria el modelo. Si bien hay casi 33% de nulos, filtro por aquellos no-nulos aunque pierda mucha información...
df = df.dropna(subset=['NewRole_binary']).copy()

# Para verificar que el mapeo coincide perfecto
print(df[['NewRole', 'NewRole_binary']].value_counts(dropna=False))

NewRole                                                                      NewRole_binary
I have neither consider or transitioned into a new career or industry        0.0               16205
I have somewhat considered changing my career and/or the industry I work in  1.0               10233
I have strongly considered changing my career and/or the industry I work in  1.0                5253
I have transitioned into a new career and/or industry voluntarily            1.0                3111
Name: count, dtype: int64


In [243]:
# Limpieza: ConvertedCompYearly ✅

# Calculamos el porcentaje de nulos de la columna
print(df['ConvertedCompYearly'].isna().mean() * 100)

# ConvertedCompYearly tiene ≈ 33.12% de nulos cuando se hace el tratamiento de la variable NewRole. Decido eliminar esas filas ya que conservo más de 20000 registros, que es un buen número a analizar.
df = df.dropna(subset=['ConvertedCompYearly']).copy()

33.12453307281191


In [244]:
# Limpieza: WorkExp ✅

# Calculamos el porcentaje de nulos de la columna
print(df['WorkExp'].isna().mean() * 100)

# WorkExp tiene ≈ 2% de nulos cuando se hace el tratamiento de las 2 variables anteriores. Al ser un número casi despreciable, simplemente elimino las filas.
df = df.dropna(subset=['WorkExp']).copy()

2.006530892841798


In [245]:
# Limpieza: JobSat ✅

# Calculamos el porcentaje de nulos de la columna
print(df['JobSat'].isna().mean() * 100)

# JobSat tiene ≈ 18.19% de nulos cuando se hace el tratamiento de las 3 variables anteriores. Decido eliminar esas filas ya que conservo más de 18500 registros, que sigue siendo un buen número a analizar.
df = df.dropna(subset=['JobSat']).copy()

17.367474898057615


In [246]:
# Transformación y limpieza: AIFrustration ✅

import numpy as np
import pandas as pd

# 1. Definimos el diccionario mapeando cada opción a su puntaje
mapeo_frustracion = {
    'AI solutions that are almost right, but not quite': 1,
    'Debugging AI-generated code is more time-consuming': 1,
    "I've become less confident in my own problem-solving": 1,
    "It's hard to understand how or why the code works": 1,
    "I haven't encountered any problems": 0,
    "I don't use AI tools regularly": 0
}                                                

# 2. Función para mapear y sumar cuando hay múltiples respuestas en una celda
def mapear_opciones_multiples(celda):
    if pd.isna(celda):
        return np.nan
    
    # Estandarizamos comillas (convierte comilla curvada ’ en estándar ')
    celda_normalizada = str(celda).replace('’', "'")
    
    # Separamos la celda por punto y coma
    opciones_seleccionadas = celda_normalizada.split(';')
    
    # Las entradas de "(write in)" generan datos ruidosos y dificiles de estandarizar por lo que buscamos cada opción en el diccionario (si no existe o es "Other", devuelve 0).
    score = sum(mapeo_frustracion.get(opcion.strip(), 0) for opcion in opciones_seleccionadas)
    
    return score

# 3. Aplicamos la transformación
df['AIFrustration_score'] = df['AIFrustration'].apply(mapear_opciones_multiples)

# 4. Verificamos los resultados
print(df['AIFrustration_score'].value_counts(dropna=False).sort_index())

# JobSat tiene ≈ 6.84% de nulos cuando se hace el tratamiento de las 4 variables anteriores. Decido eliminar esas filas ya que conservo más de 17500 registros, que sigue siendo un buen número a analizar.
df = df.dropna(subset=['AIFrustration_score']).copy()

AIFrustration_score
0.0    2645
1.0    6539
2.0    5811
3.0    2086
4.0     475
NaN    1290
Name: count, dtype: int64


In [247]:
# Transformación y limpieza: AIAcc ✅

df['AIAcc'].value_counts(dropna=False)

mapeo = {
    'Highly distrust': 1,
    'Somewhat distrust': 2,
    'Neither trust nor distrust': 3,
    'Somewhat trust': 4,
    'Highly trust': 5
}

# Hago el mapeo
df['AIAcc_score'] = df['AIAcc'].map(mapeo)


# Calculamos el porcentaje de nulos de la columna
print(df['AIAcc_score'].isna().mean() * 100)

# Como AIAcc tiene ≈ 0.14% de nulos cuando se hace el tratamiento de todas las variables anteriores. Decido eliminar esas filas ya sigo conservando más de 17500 registros, que sigue siendo un gran número a analizar.
df = df.dropna(subset=['AIAcc_score']).copy()

# Para verificar que el mapeo coincide perfecto
print(df[['AIAcc', 'AIAcc_score']].value_counts(dropna=False))

0.1367053998632946
AIAcc                       AIAcc_score
Somewhat trust              4.0            5174
Somewhat distrust           2.0            4785
Neither trust nor distrust  3.0            3673
Highly distrust             1.0            3499
Highly trust                5.0             401
Name: count, dtype: int64


In [248]:
# 1. Lista con las 6 variables finales de la hipótesis
columnas = [
    'NewRole_binary', 
    'AIFrustration_score', 
    'AIAcc_score', 
    'ConvertedCompYearly', 
    'WorkExp', 
    'JobSat'
]

# 2. Selección de columnas y eliminación de registros nulos
df_multivariada = df[columnas].copy()
df_multivariada

,NewRole_binary,AIFrustration_score,AIAcc_score,ConvertedCompYearly,WorkExp,JobSat
0,0.0,1.0,3.0,61256.0,8.0,10.0
1,1.0,2.0,3.0,104413.0,2.0,9.0
3,0.0,1.0,4.0,36197.0,4.0,6.0
4,0.0,1.0,3.0,60000.0,21.0,7.0
5,1.0,1.0,2.0,120000.0,15.0,7.0
...,...,...,...,...,...,...
49048,1.0,3.0,4.0,1053.0,23.0,5.0
49056,0.0,2.0,4.0,41765.0,21.0,9.0
49066,0.0,1.0,1.0,138470.0,19.0,6.0
49075,1.0,2.0,2.0,82366.0,4.0,2.0


##### Aclaraciones:
- No sobreescribi las columnas, cree nuevas para no perder la información. ¿Está bien?
- Conteo final: 49191 -> 17532 registros (perdí 0.35% registros del dataset).

Esta todo listo, el data set con solo esas columnas, con todo limpio y nada raro.
Quedaron 17500 registros a analizar.